<a href="https://colab.research.google.com/github/sumair789-lgtm/urdu-ocr-codesaviours-si26--Sumair-/blob/main/SI26-Week3-Sumair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this Week I am increasing my Dataset to 200 plus images ad after that I will make my dataset class and Test the class on my dataset respectively.If the shapes print without errors, your dataset is correctly wired up and ready for model
training in Week 4.

In [22]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define paths
raw_dir = '/content/drive/MyDrive/urdu-ocr-si26/data/raw'
categories = ['newspaper', 'other', 'signboards', 'synthetic', 'books']

print(" Scanning your raw folders:\n" + "="*45)
total_images = 0

for category in categories:
    category_path = os.path.join(raw_dir, category)
    if os.path.exists(category_path):
        images = [img for img in os.listdir(category_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f" {category.upper():<12} | Found: {len(images)} images")
        total_images += len(images)
    else:
        print(f" Warning: Folder '{category}' not found at {category_path}")

print("="*45)
print(f" Total raw images across all categories: {total_images}")
if total_images >= 200:
    print(" Excellent! I have met the Week 3 goal of 200+ images.")
else:
    print(f" You currently have {total_images} images. You need {200 - total_images} more to hit your 200-image milestone!")

 Scanning your raw folders:
 NEWSPAPER    | Found: 60 images
 OTHER        | Found: 81 images
 SIGNBOARDS   | Found: 42 images
 SYNTHETIC    | Found: 40 images
 BOOKS        | Found: 50 images
 Total raw images across all categories: 273
 Excellent! I have met the Week 3 goal of 200+ images.


In [23]:
import os
import shutil
from PIL import Image
from google.colab import drive

# Step 1 — Connect Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Step 2 — Define Paths
raw_dir = '/content/drive/MyDrive/urdu-ocr-si26/data/raw'
processed_dir = '/content/drive/MyDrive/urdu-ocr-si26/data/processed'
categories = ['newspaper', 'other', 'signboards', 'synthetic', 'books']

# Step 3 — Clean and Recreate the Processed Folder
if os.path.exists(processed_dir):
    print(" Removing the old processed folder...")
    shutil.rmtree(processed_dir)

os.makedirs(processed_dir, exist_ok=True)
print(" Created a brand-new, empty 'processed' folder.")

# Step 4 — Run Preprocessing
print("\n Starting preprocessing pipeline...\n" + "="*50)
processed_count = 0

for category in categories:
    raw_category_path = os.path.join(raw_dir, category)
    processed_category_path = os.path.join(processed_dir, category)

    if os.path.exists(raw_category_path):
        # Create corresponding subfolder in 'processed'
        os.makedirs(processed_category_path, exist_ok=True)

        # Get all images in this folder
        images = [img for img in os.listdir(raw_category_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f" Processing folder: '{category}' ({len(images)} images found)...")

        for img_name in images:
            raw_img_path = os.path.join(raw_category_path, img_name)
            processed_img_path = os.path.join(processed_category_path, img_name)

            try:
                # Open image
                with Image.open(raw_img_path) as img:
                    # Convert to RGB (essential for TrOCR)
                    img_rgb = img.convert('RGB')

                    # Resize to standard height of 384 while maintaining aspect ratio
                    # (This keeps the text readable and standardized for the model)
                    base_height = 384
                    w_percent = (base_height / float(img_rgb.size[1]))
                    w_size = int((float(img_rgb.size[0]) * float(w_percent)))
                    img_resized = img_rgb.resize((w_size, base_height), Image.Resampling.LANCZOS)

                    # Save as PNG inside 'processed'
                    img_resized.save(processed_img_path, format='PNG')
                    processed_count += 1

            except Exception as e:
                print(f"    Error processing {img_name}: {e}")

print("="*50)
print(f" Success! Re-processed and saved {processed_count} images into '{processed_dir}'")

 Removing the old processed folder...
 Created a brand-new, empty 'processed' folder.

 Starting preprocessing pipeline...
 Processing folder: 'newspaper' (60 images found)...
 Processing folder: 'other' (81 images found)...
 Processing folder: 'signboards' (42 images found)...
 Processing folder: 'synthetic' (40 images found)...
 Processing folder: 'books' (50 images found)...
 Success! Re-processed and saved 273 images into '/content/drive/MyDrive/urdu-ocr-si26/data/processed'


A Dataset class is code that tells your model: “here is one image and here is the correct
text for it.” It acts as a filing system — whenever the model asks for sample number N,
the class opens the right image, processes it, and returns it alongside its label.

Build the Dataset Class

In [5]:
!pip install transformers torch pillow pandas
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Load and convert image
        image = Image.open(row['image']).convert('RGB')

        # Process image for the model
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}

Test Dataset

In [20]:
from transformers import TrOCRProcessor, ViTImageProcessor, RobertaTokenizer
import torch
import os

# Initialize image processor and tokenizer separately
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Create dataset
dataset = UrduOCRDataset('labels.csv', processor)

# Quick alignment for the CSV column name from Excel (if present)
if 'Unnamed: 1' in dataset.data.columns:
    dataset.data = dataset.data.rename(columns={'Unnamed: 1': 'text'})

# Point image paths to your mounted Google Drive folder
drive_base_path = "/content/drive/My Drive/urdu-ocr-si26"
dataset.data['image'] = dataset.data['image'].apply(lambda x: os.path.join(drive_base_path, x) if not str(x).startswith('/content') else x)

# Test it loads correctly
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')
# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 273 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 218
Testing samples: 55
